In [1]:
import warnings

import lillestrom_utils as utils
import matplotlib.pyplot as plt
import nivapy3 as nivapy
import numpy as np
import pandas as pd
import seaborn as sn

warnings.simplefilter("ignore")
plt.style.use("ggplot")

# TEOTIL3 for Lillestrøm kommune
# Notebook 07b: Comparision with TEOTIL for waterbodies

This notebook compares results from TEOTIL3 to measured loads at specific **waterbodies** (calculated in notebook 4b).

## 1. Sites of interest

**Note:** Sagelva, Nitelva and Leira are all reasonably large catchments comprising several regine units. This makes them suitable for modelling using TEOTIL3. Rømua and Åa are both single regine catchments, which is not ideal for TEOTIL3. Rømua is larger than Åa, so I would expect results for Rømua to be better than for Åa, but neither is really suitable for modelling using TEOTIL3. Jeksla and Gansåa are both small (less than one regine) and the upstream area is very large. This makes them difficult to model using TEOTIL.

For Jeksla, Gansåa and Åa, I will use the `local` results from TEOTIL to get a rough estimate of source apportionment.

In [2]:
# Read site data
xl_path = r"../data/lillestrom_monitoring_sites.xlsx"
stn_df = pd.read_excel(xl_path, sheet_name="chem_stns")
stn_df = stn_df[stn_df["comment"].str.startswith("Downstream")]

stn_df

,catchment,station_id,station_name,regine,wb_id,wb_name,lon,lat,comment
0,Sagelva,002-46599,Sagelva ved Skjetten bro (F3),002.CBA0,002-3899-R,Fjellhamarelva - Sagelva,11.01695,59.95850,Downstream
2,Nitelva,002-30586,Rud Nitelva (N8),002.CB0,002-3891-R,Nedre Nitelva,11.05700,59.94553,Downstream
4,Leira,002-29659,Leira ved Borgen bru (L5),002.CAA0,002-3384-R,Leira nedstrøms Krokfoss,11.10032,59.95036,Downstream
6,Rømua,002-28960,Rømua ved Lørenfallet - RØM1,002.D2Z,002-3659-R,Rømua,11.22099,60.02163,Downstream
7,Jeksla,002-30593,"Jeksla ved Haugli, J14",002.CAA0,002-599-R,Jeksla,11.10637,60.00199,Downstream. Small part of regine
8,Åa,002-60929,Fossåa ved Sylta (ÅA1),002.D3Z,002-3685-R,Fossåa,11.30018,59.99209,Downstream
11,Gansåa,002-60933,Gansåa nedstrøms Dalen RA (BD),002.C5,002-3655-R,Gansåa,11.22275,59.86139,Downstream. Small part of regine


## 2. Read observed loads for waterbodies

From Notebook 04b.

In [3]:
# Observed
xl_path = r"../data/annual_loads_waterbodies.xlsx"
obs_df = pd.read_excel(xl_path, sheet_name="annual_loads")

# Catchment metadata
cat_df = pd.read_excel(r"../data/vannmiljo_catch_metadata.xlsx")

obs_df.head()

,waterbody_id,year,TOTN_tonn,TOTP_tonn,SS_tonn,TOC_tonn
0,002-3899-R,1995,43.201374,1.261320,430.532291,176.885239
1,002-3899-R,1996,42.329754,1.085266,384.791561,143.628876
2,002-3899-R,1997,30.788682,1.162052,428.468812,128.667945
3,002-3899-R,1998,36.944484,1.682400,637.184286,175.929176
4,002-3899-R,1999,60.889153,3.273468,1481.663014,332.185908


## 3. Read TEOTIL3 data

In [4]:
# TEOTIL options
st_yr = 2013
end_yr = 2024
nve_data_yr = 2025
agri_loss_model = "annual"
pars = ["TOTN", "TOTP", "TOC", "SS"]

# Which TEOTIL3 results to use for each Catchment
accum_stns = ["Sagelva", "Nitelva", "Leira", "Rømua"]
local_stns = ["Jeksla", "Gansåa", "Åa"]

In [5]:
# Two options: standard TEOTIL3 grouping and separate spredt and overflows
def get_agg_opts(agg_type, stat, par):
    if agg_type == "default":
        return {
            "Jordbruk": [f"{stat}_agriculture_{par}_tonnes"],
            "Avløp": [
                f"{stat}_large-wastewater_{par}_tonnes",
                f"{stat}_spredt_{par}_tonnes",
                f"{stat}_overflow_{par}_tonnes",
            ],
            "Industri": [f"{stat}_industry_{par}_tonnes"],
            "Bebygd": [f"{stat}_urban_{par}_tonnes"],
            "Bakgrunn": [
                f"{stat}_agriculture-background_{par}_tonnes",
                f"{stat}_upland_{par}_tonnes",
                f"{stat}_wood_{par}_tonnes",
            ],
        }
    elif agg_type == "separate":
        return {
            "Jordbruk": [f"{stat}_agriculture_{par.lower()}_tonnes"],
            "Avløp": [f"{stat}_large-wastewater_{par.lower()}_tonnes"],
            "Spredt": [f"{stat}_spredt_{par.lower()}_tonnes"],
            "Overflow": [f"{stat}_overflow_{par.lower()}_tonnes"],
            "Industri": [f"{stat}_industry_{par.lower()}_tonnes"],
            "Bebygd": [f"{stat}_urban_{par.lower()}_tonnes"],
            "Bakgrunn": [
                f"{stat}_agriculture-background_{par.lower()}_tonnes",
                f"{stat}_upland_{par.lower()}_tonnes",
                f"{stat}_wood_{par.lower()}_tonnes",
            ],
        }
    else:
        raise ValueError("agg_type' not recognised.")

In [6]:
# Output Excel file
xl_path = r"../data/teotil3_modelled_loads_waterbodies.xlsx"
with pd.ExcelWriter(xl_path) as writer:
    for agg_type in ["default", "separate"]:
        res_dict = {}
        for stat in ["accum", "local"]:
            # Get TEO3 data
            teo_df = utils.get_teotil3_results(
                st_yr,
                end_yr,
                stn_df["regine"].tolist(),
                agri_loss_model,
                nve_data_yr,
                stat=stat,
            )

            # Aggregate to desired level
            id_cols = ["regine", "Parameter", "År"]
            par_df_list = []
            for par in pars:
                agg_dict = get_agg_opts(agg_type, stat, par.lower())
                par_df = utils.aggregate_parameters(
                    teo_df,
                    par.lower(),
                    agg_dict=agg_dict,
                    stat=stat,
                )
                par_df["Parameter"] = par
                val_cols = [col for col in par_df.columns if col not in id_cols]
                par_df = par_df[id_cols + val_cols].sort_values(id_cols)
                par_df_list.append(par_df)

            # Merge
            stat_df = pd.concat(par_df_list, axis="rows")
            stat_df = pd.merge(
                stn_df[["catchment", "wb_id", "wb_name", "regine"]],
                stat_df,
                how="right",
                on="regine",
            )
            stat_df["Akvakultur"] = stat_df["Akvakultur"].fillna(0)

            # Scale to allow for regine outflows not exactly matching monitoring sites
            id_cols = ["catchment", "wb_id", "wb_name", "regine", "Parameter", "År"]
            val_cols = [col for col in stat_df.columns if col not in id_cols]
            for idx, row in stn_df.iterrows():
                stn_id = row["station_id"]
                reg_id = row["regine"]
                name = row["catchment"]

                obs_area = cat_df.query("station_code == @stn_id")["cat_area_km2"].iloc[
                    0
                ]
                if stat == "accum":
                    mod_area = teo_df.query("regine == @reg_id")[
                        "accum_upstr_area_km2"
                    ].iloc[0]
                else:
                    mod_area = teo_df.query("regine == @reg_id")[
                        "local_a_cat_land_km2"
                    ].iloc[0]
                scale_fac = obs_area / mod_area
                print(f"{name} ({stat}; {agg_type}) scale factor: {scale_fac:.3f}.")
                mask = stat_df["catchment"] == name
                stat_df.loc[mask, val_cols] *= scale_fac

            # Add to results
            res_dict[stat] = stat_df

        # Combine correct data for 'accum' and 'local'
        mod_df = pd.concat(
            [
                res_dict["accum"].query("catchment in @accum_stns"),
                res_dict["local"].query("catchment in @local_stns"),
            ],
            axis="rows",
        )

        # Save
        mod_df.to_excel(writer, sheet_name=agg_type, index=False)

Sagelva (accum; default) scale factor: 1.000.
Nitelva (accum; default) scale factor: 0.993.
Leira (accum; default) scale factor: 0.997.
Rømua (accum; default) scale factor: 0.958.
Jeksla (accum; default) scale factor: 0.023.
Åa (accum; default) scale factor: 0.982.
Gansåa (accum; default) scale factor: 0.001.
Sagelva (local; default) scale factor: 7.245.
Nitelva (local; default) scale factor: 58.395.
Leira (local; default) scale factor: 11.501.
Rømua (local; default) scale factor: 0.958.
Jeksla (local; default) scale factor: 0.263.
Åa (local; default) scale factor: 0.982.
Gansåa (local; default) scale factor: 0.364.
Sagelva (accum; separate) scale factor: 1.000.
Nitelva (accum; separate) scale factor: 0.993.
Leira (accum; separate) scale factor: 0.997.
Rømua (accum; separate) scale factor: 0.958.
Jeksla (accum; separate) scale factor: 0.023.
Åa (accum; separate) scale factor: 0.982.
Gansåa (accum; separate) scale factor: 0.001.
Sagelva (local; separate) scale factor: 7.245.
Nitelva (lo

## 4. Compare modelled to observed

In [7]:
for agg_type in ["default", "separate"]:
    if agg_type == "default":
        val_cols = ["Jordbruk", "Avløp", "Industri", "Bebygd", "Bakgrunn"]
        colour_dict = {
            # "Akvakultur": "royalblue",
            "Jordbruk": "sienna",
            "Avløp": "red",
            "Industri": "darkgrey",
            "Bebygd": "gold",
            "Bakgrunn": "limegreen",
        }
    else:
        val_cols = [
            "Jordbruk",
            "Avløp",
            "Spredt",
            "Overflow",
            "Industri",
            "Bebygd",
            "Bakgrunn",
        ]
        colour_dict = {
            # "Akvakultur": "royalblue",
            "Jordbruk": "sienna",
            "Avløp": "red",
            "Spredt": "black",
            "Overflow": "blue",
            "Industri": "darkgrey",
            "Bebygd": "gold",
            "Bakgrunn": "limegreen",
        }

    # Read TEO3 results for Lillestrøm
    xl_path = r"../data/teotil3_modelled_loads_waterbodies.xlsx"
    mod_df = pd.read_excel(xl_path, sheet_name=agg_type)

    # Get modelled and observed data for station/waterbody
    for idx, row in stn_df.iterrows():
        stn_id = row["station_id"]
        wb_id = row["wb_id"]
        name = row["catchment"]
        mod_stn_df = mod_df.query("wb_id == @wb_id").copy()
        obs_stn_df = obs_df.query("(waterbody_id == @wb_id) and (year >= 2013)").copy()

        # Plot
        fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 6))
        axes = axes.flatten()
        handles, labels = None, None
        for ax_idx, par in enumerate(pars):
            ax = axes[ax_idx]
            mod_par_df = (
                mod_stn_df.query("Parameter == @par")[["År"] + val_cols]
                .sort_values("År")
                .set_index("År")
            )
            obs_par_df = (
                obs_stn_df[["year", f"{par}_tonn"]]
                .dropna(subset=f"{par}_tonn")
                .sort_values("year")
                .set_index("year")
            )

            # Stacked bar chart for modelled
            bottom = None
            for col in val_cols:
                values = mod_par_df[col]
                if bottom is None:
                    ax.bar(
                        mod_par_df.index,
                        values,
                        label=col,
                        color=colour_dict[col],
                    )
                    bottom = values.copy()
                else:
                    ax.bar(
                        mod_par_df.index,
                        values,
                        bottom=bottom,
                        label=col,
                        color=colour_dict[col],
                    )
                    bottom += values

            # Line chart for observed
            ax.plot(
                obs_par_df.index,
                obs_par_df[f"{par}_tonn"],
                color="k",
                marker="o",
                linestyle="-",
                label="Observert",
                linewidth=2,
            )

            ax.set_title(par)
            ax.set_ylabel("Tilførsler (tonn)")
            if handles is None:
                handles, labels = ax.get_legend_handles_labels()

        # Legend
        fig.legend(
            handles,
            labels,
            loc="lower center",
            ncol=3,
            frameon=False,
        )
        plt.tight_layout(rect=[0, 0.08, 1, 0.95])

        # Save
        png_path = (
            f"../plots/modelled/mod_vs_obs_{agg_type}_{name}_waterbody_{wb_id}.png"
        )
        plt.savefig(png_path, dpi=200, bbox_inches="tight")

        plt.close()

## 5. Source apportionment

In [8]:
# Period for percentage contributions
src_st_yr = 2020
src_end_yr = 2024

plot_order = ["TOTN", "TOTP", "TOC", "SS"]

In [9]:
for agg_type in ["default", "separate"]:
    if agg_type == "default":
        val_cols = ["Jordbruk", "Avløp", "Industri", "Bebygd", "Bakgrunn"]
        colour_dict = {
            # "Akvakultur": "royalblue",
            "Jordbruk": "sienna",
            "Avløp": "red",
            "Industri": "darkgrey",
            "Bebygd": "gold",
            "Bakgrunn": "limegreen",
        }
    else:
        val_cols = [
            "Jordbruk",
            "Avløp",
            "Spredt",
            "Overflow",
            "Industri",
            "Bebygd",
            "Bakgrunn",
        ]
        colour_dict = {
            # "Akvakultur": "royalblue",
            "Jordbruk": "sienna",
            "Avløp": "red",
            "Spredt": "black",
            "Overflow": "blue",
            "Industri": "darkgrey",
            "Bebygd": "gold",
            "Bakgrunn": "limegreen",
        }

    # Read TEO3 results for Lillestrøm
    xl_path = r"../data/teotil3_modelled_loads_waterbodies.xlsx"
    mod_df = pd.read_excel(xl_path, sheet_name=agg_type).query(
        "@src_st_yr <= `År` <= @src_end_yr"
    )

    source_cols = list(colour_dict.keys())
    for wb_id, wb_df in mod_df.groupby("wb_id"):
        name = wb_df.iloc[0]["catchment"]

        # Convert to percentages
        plot_df = wb_df.drop(columns="År").groupby("Parameter")[source_cols].sum()
        plot_df = 100 * plot_df.div(plot_df.sum(axis=1), axis=0)
        plot_df = plot_df.reindex(plot_order[::-1])

        # Plot
        fig, ax = plt.subplots(figsize=(10, 0.2 * len(plot_df) + 2))
        left = 0
        for source in source_cols:
            bars = ax.barh(
                plot_df.index,
                plot_df[source],
                left=left,
                color=colour_dict[source],
                label=source,
            )
            ax.bar_label(
                bars,
                labels=[f"{v:.0f}%" if v >= 3 else "" for v in plot_df[source]],
                label_type="center",
                color="white",
                fontsize=10,
                fontweight="bold",
            )
            left += plot_df[source]

        ax.set_xlim(0, 100)
        ax.set_xlabel("Kildefordeling (%)")
        ax.legend(
            loc="center left",
            bbox_to_anchor=(1.02, 0.5),
            borderaxespad=0,
        )
        plt.tight_layout()

        # Save
        png_path = f"../plots/modelled/src_app_{agg_type}_{name}_waterbody_{wb_id}.png"
        plt.savefig(png_path, dpi=200, bbox_inches="tight")
        plt.close()